In [3]:
from pymongo import MongoClient

# 1. Conectar al contenedor de Mongo
client = MongoClient('mongodb://mongodb:27017/')

# 2. Crear (o usar) la base de datos y la colección
db = client['JuegosBigData'] 
coleccion = db['JugandoConPaginas'] # Ideal el nombre del grupo ejem: 'G_Ecommerce_scraping'

print("Conexión exitosa a MongoDB")

Conexión exitosa a MongoDB


In [2]:
import os
import time
from pymongo import MongoClient
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# -------------------------------
# LIMPIEZA
# -------------------------------
os.system("pkill -9 chrome")
os.system("pkill -9 chromedriver")

# -------------------------------
# VARIABLES
# -------------------------------
NOMBRE_GRUPO = "G9-Agrotech"
INTEGRANTE = "Mathieus Villavicencio"
ANIO = 2025

datos_finales = []

# lista de meses reales
meses = [
    "Enero","Febrero","Marzo","Abril","Mayo","Junio",
    "Julio","Agosto","Septiembre","Octubre","Noviembre","Diciembre"
]

# -------------------------------
# CONFIG SELENIUM
# -------------------------------
options = Options()
options.binary_location = "/usr/bin/google-chrome"
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--disable-gpu")
options.add_argument("--window-size=1920,1080")

driver = None

try:
    driver = webdriver.Chrome(options=options)
    print("Navegador iniciado correctamente")

    url = "https://climatologia.meteochile.gob.cl/application/anual/indiceUvbMaximoAnual/290004/2025"
    driver.get(url)

    print("Página cargando...")

    # Espera real
    WebDriverWait(driver, 30).until(
        EC.presence_of_element_located((By.CSS_SELECTOR, "table"))
    )

    time.sleep(5)

    # Scroll
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    time.sleep(3)

    # -------------------------------
    # CAPTURA DE DATOS
    # -------------------------------
    filas = driver.find_elements(By.CSS_SELECTOR, "table tbody tr")
    print("Filas encontradas:", len(filas))

    for fila in filas:
        columnas = fila.find_elements(By.TAG_NAME, "td")

        # 13 columnas 
        if len(columnas) >= 13:

            try:
                dia = int(columnas[0].text.strip())
            except:
                continue

            # recorrer los 12 meses
            for i in range(1, 13):
                valor_texto = columnas[i].text.strip()

                if valor_texto:
                    valor_limpio = valor_texto.replace(",", ".").strip()

                    try:
                        valor_float = float(valor_limpio)
                    except:
                        valor_float = None

                    datos_finales.append({
                        "grupo": NOMBRE_GRUPO,
                        "integrante": INTEGRANTE,
                        "etiqueta": "Indice UV",
                        "anio": ANIO,
                        "mes": meses[i-1],  
                        "dia": dia,
                        "valor": valor_float,
                        "variacion_decimal": None,
                        "fuente": "MeteoChile",
                        "fecha_captura": time.strftime("%Y-%m-%d %H:%M:%S")
                    })

    print(f"Extracción terminada: {len(datos_finales)} registros.")

    # DEBUG
    for d in datos_finales[:5]:
        print(d)

except Exception as e:
    print("Error en Selenium:", e)

finally:
    if driver:
        driver.quit()

# -------------------------------
# GUARDAR EN MONGO
# -------------------------------
try:
    client = MongoClient("database", 27017)
    db = client["ClimaBigData"]
    coleccion = db["MeteoChileUV"]

    if datos_finales:
        coleccion.insert_many(datos_finales)
        print("Datos cargados en MongoDB correctamente.")
    else:
        print("No hay datos para guardar.")

except Exception as e:
    print("Error en MongoDB:", e)

Navegador iniciado correctamente
Página cargando...
Filas encontradas: 44
Extracción terminada: 365 registros.
{'grupo': 'G9-Agrotech', 'integrante': 'Mathieus Villavicencio', 'etiqueta': 'Indice UV', 'anio': 2025, 'mes': 'Enero', 'dia': 1, 'valor': 10.0, 'variacion_decimal': None, 'fuente': 'MeteoChile', 'fecha_captura': '2026-04-19 22:35:00'}
{'grupo': 'G9-Agrotech', 'integrante': 'Mathieus Villavicencio', 'etiqueta': 'Indice UV', 'anio': 2025, 'mes': 'Febrero', 'dia': 1, 'valor': 13.0, 'variacion_decimal': None, 'fuente': 'MeteoChile', 'fecha_captura': '2026-04-19 22:35:00'}
{'grupo': 'G9-Agrotech', 'integrante': 'Mathieus Villavicencio', 'etiqueta': 'Indice UV', 'anio': 2025, 'mes': 'Marzo', 'dia': 1, 'valor': 10.0, 'variacion_decimal': None, 'fuente': 'MeteoChile', 'fecha_captura': '2026-04-19 22:35:00'}
{'grupo': 'G9-Agrotech', 'integrante': 'Mathieus Villavicencio', 'etiqueta': 'Indice UV', 'anio': 2025, 'mes': 'Abril', 'dia': 1, 'valor': 7.0, 'variacion_decimal': None, 'fuente'